# 19 — Invoice Extractor Agent

This notebook wires together `InvoiceExtractorTool` + `ReActAgent` to process real scanned
invoice TIF files from `public/` and return structured extracted data.

### What we'll cover

| Section | What it shows |
|---|---|
| 1 | Install check — verify pdfplumber, Pillow, pytesseract are available |
| 2 | Tool basics — call `InvoiceExtractorTool` directly on real TIF invoices |
| 3 | Selective frame extraction — extract specific frames from multi-frame TIFs |
| 4 | OCR quality inspection — character/word counts across all real files |
| 5 | Agent integration — `ReActAgent` decides when+how to call the tool |
| 6 | Batch processing — extract from both real TIF invoices in one agent turn |
| 7 | Error handling — missing file, wrong format, out-of-range frame |

> **Install**: optional `pdf` dependency group: `uv sync --group pdf`
> Tesseract OCR binary must also be installed (`choco install tesseract` / `apt install tesseract-ocr`).


In [1]:
import sys
sys.path.insert(0, "..")

# Verify PDF + OCR deps
import importlib, shutil

for pkg in ["pdfplumber", "pypdf", "PIL", "pytesseract"]:
    found = importlib.util.find_spec(pkg) is not None
    print(f"  {'✓' if found else '✗'} {pkg}")

tesseract_bin = shutil.which("tesseract")
print(f"\n  tesseract binary: {tesseract_bin or 'NOT FOUND — OCR will fail'}")
print("\nInstall everything with: uv sync --group pdf")
print("Tesseract binary: https://github.com/UB-Mannheim/tesseract/wiki (Windows)")


  ✓ pdfplumber
  ✓ pypdf
  ✓ PIL
  ✓ pytesseract

  tesseract binary: C:\Program Files\Tesseract-OCR\tesseract.EXE

Install everything with: uv sync --group pdf
Tesseract binary: https://github.com/UB-Mannheim/tesseract/wiki (Windows)


## 1. Install Check & Tool Setup

`InvoiceExtractorTool` now handles **both** PDFs and scanned images:

| File type | Engine |
|---|---|
| `.pdf` | pdfplumber (text layer + table detection) |
| `.tif` / `.tiff` / `.png` / `.jpg` | Pillow + pytesseract OCR |


In [2]:
import asyncio
from pathlib import Path
from raavan.catalog.tools.invoice_extractor.tool import InvoiceExtractorTool

tool = InvoiceExtractorTool()
print(f"name     : {tool.name}")
print(f"category : {tool.category}")
print(f"tags     : {tool.tags}")
print(f"aliases  : {tool.aliases}")


name     : invoice_extractor
category : data/extraction
tags     : ['invoice', 'pdf', 'tif', 'ocr', 'extract', 'table', 'document', 'image']
aliases  : ['pdf_extractor', 'extract_invoice', 'image_extractor']


## 2. Real Invoice Scans — TIF Files

The `public/` folder contains two real scanned invoice TIF files.
We read them directly — no synthetic data.


In [13]:
import tempfile
from pathlib import Path
from raavan.catalog.tools.invoice_extractor.tool import InvoiceExtractorTool

TMP = Path(tempfile.gettempdir())
PUBLIC = Path("../public")  # raavan/public/

tool = InvoiceExtractorTool()

tif_files = sorted(PUBLIC.glob("*.tif")) + sorted(PUBLIC.glob("*.tiff"))
png_files = sorted(PUBLIC.glob("*.png"))
print(f"Found {len(tif_files)} TIF file(s): {[f.name for f in tif_files]}\n")
print(f"Found {len(png_files)} PNG file(s): {[f.name for f in png_files]}\n")

for tif in tif_files:
    print(f"{'='*60}")
    print(f"Processing: {tif.name}")
    print(f"{'='*60}")
    result = await tool.execute(file_path=str(tif))
    if result.is_error:
        print(f"ERROR: {result.content[0]['text']}")
    else:
        print(result.content[0]["text"])
        print(f"\nMetadata: {result.app_data}")
    print()

for png in png_files:
    print(f"{'='*60}")
    print(f"Processing: {png.name}")
    print(f"{'='*60}")
    result = await tool.execute(file_path=str(png))
    if result.is_error:
        print(f"ERROR: {result.content[0]['text']}")
    else:
        print(result.content[0]["text"])
        print(f"\nMetadata: {result.app_data}")
    print()

Found 2 TIF file(s): ['0000056826.tif', '0000083797.tif']

Found 1 PNG file(s): ['Retail-Invoice-Template.png']

Processing: 0000056826.tif
--- Page 1 ---
(° . ANCHOR PLASTICS COMPANY

‘A Division of VOPLEX CORP.
36.25 38th STREET, LONG ISLANO CITY. Ny
TELEPHONE: 212 729.1494

MFG. LOCATION:
1455 IMLAY CITY RD,
LAPEER, MICH, 48446

pare January 21, 1983 QUOTATION

custom
PRECISION
Brown & Williamson Tobacco Corp. EXTRUSION
1600 W. Hill Street
Louisville, KY 40232
it ; Mr. Elmer Litzinger QUOTATION N
: AQ; Z
Your Ret, Mo, 2508 SBS
On ALL COnmesPonDENce.
T6086 Natural Aeroflex Tubular Configuration per our

Drawing PR-5258 supplied 108MM lengths stack packed.

100,000 pieces - $3.30/M pes.

Tooling and Development - $3,000.00 On

initial samples

smippinc pate_@PPFOX. 5 weeks  ¢95, Lapeer, Mich. Terms: Net _30 days
SBS / he AFTER RECEIPT OF ORDER TOOLS & DEVELOPMENT WET

‘THIS QUOTATION IS BEING FURNISHED FOR THE PURPOSE OF SUPPLYING INFORMATION RELATIVE TO PRICES, TERMS

DESCRIPTION IN 

## 3. Selective Frame Extraction

Multi-frame TIFFs work like multi-page PDFs — extract only the frames you need.


In [4]:
from PIL import Image

# How many frames does each TIF have?
for tif in tif_files:
    img = Image.open(tif)
    frames = 0
    try:
        while True:
            frames += 1
            img.seek(img.tell() + 1)
    except EOFError:
        pass
    print(f"{tif.name}: {frames} frame(s), size={img.size}, mode={img.mode}")

# Extract only frame 0 of the first TIF
first_tif = tif_files[0]
result_frame0 = await tool.execute(file_path=str(first_tif), pages=[0])
print(f"\n=== {first_tif.name} — frame 0 only ===")
print(result_frame0.content[0]["text"][:600])
print("Metadata:", result_frame0.app_data)


0000056826.tif: 1 frame(s), size=(774, 1000), mode=L
0000083797.tif: 1 frame(s), size=(762, 1000), mode=L

=== 0000056826.tif — frame 0 only ===
--- Page 1 ---
(° . ANCHOR PLASTICS COMPANY

‘A Division of VOPLEX CORP.
36.25 38th STREET, LONG ISLANO CITY. Ny
TELEPHONE: 212 729.1494

MFG. LOCATION:
1455 IMLAY CITY RD,
LAPEER, MICH, 48446

pare January 21, 1983 QUOTATION

custom
PRECISION
Brown & Williamson Tobacco Corp. EXTRUSION
1600 W. Hill Street
Louisville, KY 40232
it ; Mr. Elmer Litzinger QUOTATION N
: AQ; Z
Your Ret, Mo, 2508 SBS
On ALL COnmesPonDENce.
T6086 Natural Aeroflex Tubular Configuration per our

Drawing PR-5258 supplied 108MM lengths stack packed.

100,000 pieces - $3.30/M pes.

Tooling and Development - $3,000.00 On

in
Metadata: {'file': '..\\public\\0000056826.tif', 'format': 'image_ocr', 'total_pages': 1, 'pages_extracted': 1, 'truncated': False}


In [5]:
# Extract both TIFs and compare
second_tif = tif_files[1]
result_second = await tool.execute(file_path=str(second_tif), pages=[0])
print(f"=== {second_tif.name} — frame 0 ===")
print(result_second.content[0]["text"][:600])
print("Metadata:", result_second.app_data)


=== 0000083797.tif — frame 0 ===
--- Page 1 ---
QUOTATION

w

Atay ME. oLex purge

Tnatsaapoaie,t 46250

a] 26720 = avtomnese sanptor = 99 5: 16] nu] av [¢ 2490.00) 63400.00
opt. 102'- not fox tavsero oLaa. we we
fept. 041 - conteod Hodete 3190.00] 1200.00,
‘opt. 069 - ror 5760 new we we

Injection pe
Ey
g
D
DB
@
ra
Metadata: {'file': '..\\public\\0000083797.tif', 'format': 'image_ocr', 'total_pages': 1, 'pages_extracted': 1, 'truncated': False}


## 4. OCR Quality Inspection

Check character counts and content density across all real TIF files.


In [6]:
from pathlib import Path

for tif in tif_files:
    r = await tool.execute(file_path=str(tif))
    raw = r.content[0]["text"]
    # Split on the per-frame separator the tool emits
    frames = [s for s in raw.split("--- Page ") if s.strip()]
    print(f"\n=== {tif.name} ===")
    for i, frame_text in enumerate(frames, 1):
        # Strip the leading frame number line if present
        body = frame_text.split("\n", 1)[-1]
        chars = len(body.strip())
        words = len(body.split())
        print(f"  frame {i}: {chars} chars, {words} words")
        if chars:
            print(f"  preview: {body.strip()[:120]!r}")



=== 0000056826.tif ===
  frame 1: 2976 chars, 482 words
  preview: '(° . ANCHOR PLASTICS COMPANY\n\n‘A Division of VOPLEX CORP.\n36.25 38th STREET, LONG ISLANO CITY. Ny\nTELEPHONE: 212 729.149'

=== 0000083797.tif ===
  frame 1: 269 chars, 53 words
  preview: 'QUOTATION\n\nw\n\nAtay ME. oLex purge\n\nTnatsaapoaie,t 46250\n\na] 26720 = avtomnese sanptor = 99 5: 16] nu] av [¢ 2490.00) 634'


## 5. ReActAgent Integration

Now the tool is given to a `ReActAgent`. The agent receives a natural-language request,
decides to call `invoice_extractor`, and returns a summary.

> Unlike sections 2-4, real LLM calls happen here. Set `OPENAI_API_KEY` in `.env`.


In [19]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from raavan.integrations.llm.openai.openai_client import OpenAIClient
from raavan.core.agents.react_agent import ReActAgent
from raavan.core.memory import UnboundedMemory
from raavan.core.context import UnboundedContext

from pydantic import BaseModel

class Headers(BaseModel):
    invoice_id: str
    vendor: str
    reciver: str
    net: float
    tax: float
    total: float

class LineItem(BaseModel):
    description: str
    quantity: float
    unit_price: float
    total_price: float

class Invoice(BaseModel):
    headers: Headers
    line_items: list[LineItem]

client = OpenAIClient(api_key=os.environ["OPENAI_API_KEY"])
memory = UnboundedMemory()
context = UnboundedContext()
extractor_tool = InvoiceExtractorTool()

agent = ReActAgent(
    name='doc_agent',
    description='extracts info from docs',
    model_client=client,
    memory=memory,
    model_context=context,
    tools=[extractor_tool],
    system_instructions=(
        "You are an invoice processing assistant. "
        "When given a scanned image path, use the invoice_extractor tool to read it "
        "and then summarise: invoice number, vendor, line items, and total due."
    ),
    max_iterations=5,
)

response = await agent.run(
    f"Please extract and summarise this invoice: {png_files[0]}",

    response_schema=Invoice,
 )

print("Output text:\n")
print(response.output_text)
print("\nStructured output:\n")
print(response.structured_output.parsed if response.structured_output else None)

[doc_agent] Starting run: Please extract and summarise this invoice: ..\public\Retail-Invoice-Template.png...
[doc_agent] Step 1: tool calls → ['invoice_extractor']
[doc_agent] Executing invoice_extractor({'file_path': '..\\public\\Retail-Invoice-Template.png', 'pages': [], 'extract_tables': True})
[doc_agent] Step 2: tool calls → ['invoice_extractor']
[doc_agent] Executing invoice_extractor({'file_path': '..\\public\\Retail-Invoice-Template.png', 'pages': [0], 'extract_tables': True})
[doc_agent] Step 3: final answer
Output text:

{
  "headers": {
    "invoice_id": "INV-000002",
    "vendor": "Zylker Thread & Weave",
    "reciver": "Scott, Melba R., 2476 Blackwell Street, Fairbanks, 99701 Colorado, USA",
    "net": 61.97,
    "tax": 3.09,
    "total": 65.06
  },
  "line_items": [
    {
      "description": "Pepe Jeans - Tapered fit Mid rise - Blue",
      "quantity": 1.0,
      "unit_price": 24.99,
      "total_price": 24.99
    },
    {
      "description": "Boys Shirt - Size 36, Mos

In [20]:
from IPython.display import display_html
from json2html import json2html
import json

display_html(json2html.convert(json=json.loads(response.output_text)), raw=True)

headers invoice_id INV-000002 vendor Zylker Thread & Weave reciver Scott, Melba R., 2476 Blackwell Street, Fairbanks, 99701 Colorado, USA net 61.97 tax 3.09 total 65.06 line_items description quantity unit_price total_price Pepe Jeans - Tapered fit Mid rise - Blue 1.0 24.99 24.99 Boys Shirt - Size 36, Mosaic design 1.0 16.99 16.99 Men Shirt - Size 42, Striped Blue 1.0 19.99 19.99

## 6. Batch Processing — Multiple Real Invoices in One Turn

Pass both real TIF files. The agent loops with `invoice_extractor` for each file.


In [ ]:
from pathlib import Path

paths = [str(f) for f in tif_files]
print("Files to process:", paths)

# Without LLM: call tool directly on each, collect results
print("\n=== Batch extraction (tool only, no LLM) ===")
for p in paths:
    r = await tool.execute(file_path=p)
    print(f"\n-- {Path(p).name} --")
    print(r.content[0]["text"][:400])


In [ ]:
# With LLM: ask the agent to process the whole batch in one message
from raavan.core.context import UnboundedContext

batch_memory = UnboundedMemory()
batch_context = UnboundedContext()
batch_agent = ReActAgent(
    name="batch_invoice_agent",
    description="Processes multiple invoices in one turn",
    model_client=client,
    memory=batch_memory,
    model_context=batch_context,
    tools=[extractor_tool],
    system_instructions=(
        "You are an invoice processing assistant. "
        "Extract each invoice and return a JSON array with keys: "
        "invoice_id, vendor, total."
    ),
    max_iterations=10,
)

file_list = "\n".join(f"- {f}" for f in tif_files)
batch_response = await batch_agent.run(
    f"Process all of these invoices and return a JSON summary:\n{file_list}"
)
print(batch_response)


## 7. Error Handling

The tool returns `is_error=True` (not exceptions) for every failure case — the agent can read
the error message and decide what to do next.


In [ ]:
cases = [
    ("Missing file",       {"file_path": str(TMP / "does_not_exist.tif")}),
    ("Wrong extension",    {"file_path": "pyproject.toml"}),
    ("Out-of-range frame", {"file_path": str(tif_files[0]), "pages": [999]}),
]

for label, kwargs in cases:
    r = await tool.execute(**kwargs)
    status = "ERROR" if r.is_error else "OK"
    print(f"[{status}] {label}: {r.content[0]['text'][:80]}")
